In [0]:
# install other required packages
install.packages(c("tidyverse", "tidyterra", "sf", "terra")) #, "lidaRtRee"))

In [0]:
setwd("/Workspace/Users/joseph.beesley@defra.onmicrosoft.com/ReForeSt/")
source("WP3/code/mapped_outputs/0_setup_gaps.R")
source("WP3/code/mapped_outputs/1_functions.R")

In [0]:
# read in nfi
nfi_all <- st_read(path_NFI_DASH, layer = "NFI2020_Interim_v1_WoodlandMap") %>%
  filter(IFT_IOA %in% c("Broadleaved", "Conifer", "Mixed mainly broadleaved", "Mixed mainly conifer", "Coppice", "Coppice with standards", "Low density"))

In [0]:
# read in districts
districts <- st_read("/dbfs/mnt/lab/unrestricted/joebeesley/FE_Economics_of_Resilience/data/input/SCDB/SCDB_districts.gpkg")

In [0]:
# join districts to nfi
nfi <-
  nfi_all %>%
  st_join(., districts, left = TRUE) %>% 
  vect()

In [0]:
# read in gap fraction
gap_frac <- rast(paste0(path_gap_frac_eng_DASH, "gap_fraction_30m.tif"))

exact = FALSE

In [0]:
# calculate gap fraction mean for nfi units
# intensive - 24h on 15.4 cluster
nfi_gap_frac_mean <- 
  terra::extract(x = gap_frac,
                 y = nfi,
                 fun = mean,
                 exact = FALSE,
                 na.rm = TRUE,
                 bind = TRUE) %>%
  select(-Shape_Area)

In [0]:
# write out
local_path <- "/tmp/nfi_2020_gap_fraction.gpkg"
writeVector(nfi_gap_frac_mean, local_path, overwrite = T)

file.copy(local_path, paste0(path_gap_frac_eng_DASH, "nfi_2020_gap_fraction.gpkg"))
file.remove(local_path)